In [1]:
import csv
import hashlib
import numpy as np
from gostcrypto import gosthash
from collect import collect
from nist_sts.run_tests import *

# Extract

In [2]:
collect(5000000, 'data.csv')

Найден порт: COM5 - Устройство с последовательным интерфейсом USB (COM5)
Сбор 5,000,000 отсчетов... (Ctrl+C для остановки)
  Прогресс: 5,000,192/5,000,000 (100%)

Собрано: 5,000,000 отсчетов
Время: 24.1 сек
Скорость: 207,051 отсчетов/сек
Файл сохранён: data.csv


In [3]:
def load_data(filename):
    data = []
    with open(filename, 'r') as f:
        reader = csv.reader(f)
        next(reader, None)
        for row in reader:
            if row and row[1].isdigit():
                data.append(int(row[1]))

    print(f"Загружено {len(data)} отсчетов.")
    return data

In [ ]:
_BYTE_TO_BITS = np.unpackbits(np.arange(256, dtype=np.uint8).reshape(-1, 1), bitorder='big').reshape(256, 8)


def extract_compressed(raw_data, target_bits, hash_type="sha3_256", compression_ratio=2):
    HASH_CONFIG = {
        "sha256":      {"func": lambda d: hashlib.sha256(d).digest(),                   "bits": 256},
        "sha3_256":    {"func": lambda d: hashlib.sha3_256(d).digest(),                 "bits": 256},

        "sha512":      {"func": lambda d: hashlib.sha512(d).digest(),                   "bits": 512},
        "sha3_512":    {"func": lambda d: hashlib.sha3_512(d).digest(),                 "bits": 512},

        "blake2s":     {"func": lambda d: hashlib.blake2s(d, digest_size=32).digest(),  "bits": 256},
        "blake2b":     {"func": lambda d: hashlib.blake2b(d, digest_size=64).digest(),  "bits": 512},
        
        "streebog256": {"func": lambda d: gosthash.new('streebog256', data=d).digest(), "bits": 256},
        "streebog512": {"func": lambda d: gosthash.new('streebog512', data=d).digest(), "bits": 512},
    }

    if hash_type not in HASH_CONFIG:
        raise ValueError(f"Неизвестный хеш: {hash_type}. Доступны: {list(HASH_CONFIG.keys())}")

    cfg = HASH_CONFIG[hash_type]
    hash_func, hash_bits = cfg["func"], cfg["bits"]

    output_bits_per_hash = hash_bits // compression_ratio
    output_bytes = (output_bits_per_hash + 7) // 8 

    raw_array = np.diff(np.asarray(raw_data, dtype=np.uint16))
    raw_bits = (raw_array & 1).astype(np.uint8)

    n_blocks = min(len(raw_bits) // hash_bits, 
                   (target_bits + output_bits_per_hash - 1) // output_bits_per_hash)
    
    if n_blocks == 0:
        return np.array([], dtype=np.int8)

    usable = raw_bits[:n_blocks * hash_bits].reshape(n_blocks, hash_bits)
    block_bytes = np.packbits(usable, axis=1, bitorder='big')

    total_output_bits = n_blocks * output_bits_per_hash
    result_bytes = bytearray(total_output_bits // 8 + 8) 
    offset = 0

    for i in range(n_blocks):
        if offset * 8 >= target_bits:
            break
        
        digest = hash_func(bytes(block_bytes[i]))
        
        result_bytes[offset:offset + output_bytes] = digest[:output_bytes]
        offset += output_bytes

    needed_bytes = min(offset, (target_bits + 7) // 8)
    result_array = np.frombuffer(bytes(result_bytes[:needed_bytes]), dtype=np.uint8)
    
    all_bits = _BYTE_TO_BITS[result_array].ravel()

    return all_bits[:target_bits].astype(np.int8)

In [21]:
data = load_data('data.csv')
extracted_bits = extract_compressed(data, 1000000, 'sha3_256', 5)
run_tests(extracted_bits)

Загружено 5000000 отсчетов.
Длина последовательности: 1000000 бит
Распределение: 0.5001 (1) / 0.4999 (0)
----------------------------------------------------------------------
Название теста                                | P-value    | Статус
----------------------------------------------------------------------
1. Frequency (Monobit) Test                   | 0.801041   | Пройден
2. Block Frequency Test                       | 0.965655   | Пройден
3. Runs Test                                  | 0.040554   | Пройден
4. Longest Run of Ones in a Block             | 0.644308   | Пройден
5. Binary Matrix Rank Test                    | 0.193681   | Пройден
6. Spectral (DFT) Test                        | 0.663506   | Пройден
7. Non-overlapping Template (mean P)          | 0.687779   | Пройден
8. Overlapping Template Matching (m=9)        | 0.857777   | Пройден
9. Maurer's Universal Test (L=6)              | 0.912145   | Пройден
10. Linear Complexity Test                    | 0.370076   | Про

In [23]:
data = load_data('data.csv')
extracted_bits = extract_compressed(data, 1000000, 'blake2s', 5)
run_tests(extracted_bits)

Загружено 5000000 отсчетов.
Длина последовательности: 1000000 бит
Распределение: 0.4996 (1) / 0.5004 (0)
----------------------------------------------------------------------
Название теста                                | P-value    | Статус
----------------------------------------------------------------------
1. Frequency (Monobit) Test                   | 0.458087   | Пройден
2. Block Frequency Test                       | 0.126171   | Пройден
3. Runs Test                                  | 0.997965   | Пройден
4. Longest Run of Ones in a Block             | 0.113697   | Пройден
5. Binary Matrix Rank Test                    | 0.385080   | Пройден
6. Spectral (DFT) Test                        | 0.907276   | Пройден
7. Non-overlapping Template (mean P)          | 0.546191   | Пройден
8. Overlapping Template Matching (m=9)        | 0.275292   | Пройден
9. Maurer's Universal Test (L=6)              | 0.876974   | Пройден
10. Linear Complexity Test                    | 0.132143   | Про

In [24]:
data = load_data('data.csv')
extracted_bits = extract_compressed(data, 1000000, 'streebog256', 5)
run_tests(extracted_bits)

Загружено 5000000 отсчетов.
Длина последовательности: 1000000 бит
Распределение: 0.5002 (1) / 0.4998 (0)
----------------------------------------------------------------------
Название теста                                | P-value    | Статус
----------------------------------------------------------------------
1. Frequency (Monobit) Test                   | 0.745938   | Пройден
2. Block Frequency Test                       | 0.076214   | Пройден
3. Runs Test                                  | 0.382175   | Пройден
4. Longest Run of Ones in a Block             | 0.555764   | Пройден
5. Binary Matrix Rank Test                    | 0.991202   | Пройден
6. Spectral (DFT) Test                        | 0.373843   | Пройден
7. Non-overlapping Template (mean P)          | 0.551227   | Пройден
8. Overlapping Template Matching (m=9)        | 0.356539   | Пройден
9. Maurer's Universal Test (L=6)              | 0.607804   | Пройден
10. Linear Complexity Test                    | 0.327913   | Про